### ROW REMOVAL 


In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.compose import ColumnTransformer
import os

import import_ipynb
import importlib
import functions as fc
importlib.reload(fc)

print(os.getcwd())

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


/Users/elif/Desktop/Projects/privacy_in_ml


In [6]:
cc_train = pd.read_csv("datasets/creditcard_train.csv")
cc_test = pd.read_csv("datasets/creditcard_test.csv")

def get_dummies_all(X_train, X_test):
    """
    Converts all categorical variables in a DataFrame into dummy (one-hot encoded) variables.
    """
    categorical_cols = ["MARRIAGE", "SEX", "EDUCATION"]
    X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
    X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)
    return X_train_encoded, X_test_encoded

X_train = cc_train.drop('default.payment.next.month', axis=1)
X_test = cc_test.drop('default.payment.next.month', axis=1)
y_train = cc_train['default.payment.next.month']
y_test = cc_test['default.payment.next.month']

X_train, X_test = get_dummies_all(X_train, X_test)


In [7]:
numeric_features = [
    "LIMIT_BAL",
    "AGE",
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6",
    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6",
]

ordinal_features = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

def preprocess(X_tr, X_te):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("ord", StandardScaler(), ordinal_features)
        ],
        remainder="passthrough"
    )
    X_tr_sc = pd.DataFrame(
        preprocessor.fit_transform(X_tr),
        columns=preprocessor.get_feature_names_out(),
        index=X_tr.index
    )
    X_te_sc = pd.DataFrame(
        preprocessor.transform(X_te),
        columns=preprocessor.get_feature_names_out(),
        index=X_te.index
    )
    return X_tr_sc, X_te_sc


### Run the row-removal 

In [8]:
n_trials = 50
np.random.seed(42)

output_file = "results/row_removal_cc.xlsx"

#Baseline
X_train_sc, X_test_sc = preprocess(X_train, X_test)

model = LogisticRegression(C=1, max_iter=1000, random_state=42)
model.fit(X_train_sc, y_train)

print("Training set (full data): ")
fc.predict_binary(model, X_train_sc, y_train, conf_matrix=False)

fc.predict_binary_save_results(
    model,
    X_test_sc,
    y_test,
    conf_matrix=False,
    perturbation_type="RowRemoval",
    epsilon=0,
    row_id=None,
    output_file=output_file
)

#Trials
for trial in range(1, n_trials + 1):
    idx = np.random.randint(0, X_train.shape[0])
    removed_label = X_train.index[idx]

    X_train_reduced = X_train.drop(index=removed_label)
    y_train_reduced = y_train.drop(index=removed_label)

    X_train_reduced_sc, X_test_reduced_sc = preprocess(X_train_reduced, X_test)

    trial_model = LogisticRegression(C=1, max_iter=1000, random_state=42)
    trial_model.fit(X_train_reduced_sc, y_train_reduced)

    print(f"Trial {trial}/{n_trials} - removed row {removed_label}")
    print("Training set: ")
    fc.predict_binary(trial_model, X_train_reduced_sc, y_train_reduced, conf_matrix=False)

    fc.predict_binary_save_results(
        trial_model,
        X_test_reduced_sc,
        y_test,
        conf_matrix=False,
        perturbation_type="RowRemoval",
        epsilon=trial,
        row_id=removed_label,
        output_file=output_file
    )


/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training set (full data): 
---------------------------------------
Accuracy: 0.8105
Precision: 0.7137
Recall: 0.2431
F1 Score: 0.3627
---------------------------------------
---------------------------------------
Accuracy: 0.8103
Precision: 0.7002
Recall: 0.2331
F1 Score: 0.3497
---------------------------------------
Saved results to results/row_removal_cc.xlsx


/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Trial 1/50 - removed row 23654
Training set: 
---------------------------------------
Accuracy: 0.8109
Precision: 0.7170
Recall: 0.2437
F1 Score: 0.3637
---------------------------------------
---------------------------------------
Accuracy: 0.8100
Precision: 0.6998
Recall: 0.2308
F1 Score: 0.3471
---------------------------------------
Saved results to results/row_removal_cc.xlsx


/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


...

/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Trial 48/50 - removed row 10253
Training set: 
---------------------------------------
Accuracy: 0.8103
Precision: 0.7148
Recall: 0.2410
F1 Score: 0.3605
---------------------------------------
---------------------------------------
Accuracy: 0.8110
Precision: 0.7039
Recall: 0.2353
F1 Score: 0.3527
---------------------------------------
Saved results to results/row_removal_cc.xlsx


/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Trial 49/50 - removed row 8433
Training set: 
---------------------------------------
Accuracy: 0.8112
Precision: 0.7172
Recall: 0.2454
F1 Score: 0.3656
---------------------------------------
---------------------------------------
Accuracy: 0.8100
Precision: 0.6961
Recall: 0.2338
F1 Score: 0.3501
---------------------------------------
Saved results to results/row_removal_cc.xlsx
Trial 50/50 - removed row 10233
Training set: 
---------------------------------------
Accuracy: 0.8108
Precision: 0.7174
Recall: 0.2427
F1 Score: 0.3627
---------------------------------------
---------------------------------------
Accuracy: 0.8102
Precision: 0.7005
Recall: 0.2315
F1 Score: 0.3480
---------------------------------------
Saved results to results/row_removal_cc.xlsx


/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
